# Film Success Prediction (Capstone Final Project)
# Step 1: Data Preparation

## Business problem

Studios and business stakeholders want to make better *up-front* decisions (before release) about which films are likely to:
1. **Generate high revenue** (financial success), and
2. **Have staying power** (strong audience engagement and positive memory over time).

Machine learning models are built to predict these outcomes using structured film metadata (budget, genre, runtime, release year, ratings / votes, etc.) and turns results into actionable recommendations.


## Data sources

I am using:
- **IMDb Non-Commercial Datasets** (title basics + ratings).  
- **TMDB 5000 Movies dataset** (commonly used in ML projects; includes budget, revenue, genres, votes).



In [24]:
# Core
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Viz
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Sklearn
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay
)

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

PROJECT_DIR = Path(".").resolve()
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)


## 1) Download data (if missing)

In [25]:
import requests

def download_if_missing(url: str, out_path: Path, chunk_size: int = 1024 * 1024) -> Path:
    """Download a file if it doesn't exist locally."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"Found: {out_path} ({out_path.stat().st_size/1e6:.1f} MB)")
        return out_path

    print(f"Downloading to {out_path} ...")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    print(f"Download complete: {out_path} ({out_path.stat().st_size/1e6:.1f} MB)")
    return out_path

# IMDb (official dataset distribution)
IMDB_BASICS_URL = "https://datasets.imdbws.com/title.basics.tsv.gz"
IMDB_RATINGS_URL = "https://datasets.imdbws.com/title.ratings.tsv.gz"

basics_gz = download_if_missing(IMDB_BASICS_URL, DATA_DIR / "title.basics.tsv.gz")
ratings_gz = download_if_missing(IMDB_RATINGS_URL, DATA_DIR / "title.ratings.tsv.gz")

# TMDB 5000 (public mirror of the same CSV file name)
TMDB_MOVIES_URL = "https://gist.githubusercontent.com/cicerojmm/f95a54d4f76de3c84415d0f703aa7e3c/raw/fdcf8081fd4178b3be574c0059bb52aae3b840b3/tmdb_5000_movies.csv"
tmdb_csv = download_if_missing(TMDB_MOVIES_URL, DATA_DIR / "tmdb_5000_movies.csv")


Found: /content/data/title.basics.tsv.gz (216.1 MB)
Found: /content/data/title.ratings.tsv.gz (8.2 MB)
Found: /content/data/tmdb_5000_movies.csv (5.7 MB)


## 2) Load IMDb basics database + ratings and filter to feature films

In [26]:

# IMDb title basics (TSV)
df_basics = pd.read_csv(basics_gz, sep="\t", compression="gzip", low_memory=False)

# Keep feature films (movie) and drop obvious null markers (\N)
df_basics = df_basics.replace({r"\N": np.nan})
df_basics = df_basics[df_basics["titleType"] == "movie"].copy()

# Cast year/runtime to numeric
df_basics["startYear"] = pd.to_numeric(df_basics["startYear"], errors="coerce")
df_basics["runtimeMinutes"] = pd.to_numeric(df_basics["runtimeMinutes"], errors="coerce")

df_basics = df_basics[["tconst", "primaryTitle", "originalTitle", "startYear", "runtimeMinutes", "genres"]]

# IMDb ratings
df_ratings = pd.read_csv(ratings_gz, sep="\t", compression="gzip")
df_ratings = df_ratings.replace({r"\N": np.nan})
df_ratings["averageRating"] = pd.to_numeric(df_ratings["averageRating"], errors="coerce")
df_ratings["numVotes"] = pd.to_numeric(df_ratings["numVotes"], errors="coerce")

# Merge
df_imdb = df_basics.merge(df_ratings, on="tconst", how="inner")
df_imdb.head()


,tconst,primaryTitle,originalTitle,startYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0000009,Miss Jerry,Miss Jerry,1894.0,45.0,Romance,5.2,232
1,tt0000147,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,1897.0,100.0,"Documentary,News,Sport",5.3,584
2,tt0000335,Soldiers of the Cross,Soldiers of the Cross,1900.0,40.0,"Biography,Drama",5.5,65
3,tt0000502,Bohemios,Bohemios,1905.0,100.0,NaN,3.1,26
4,tt0000574,The Story of the Kelly Gang,The Story of the Kelly Gang,1906.0,70.0,"Action,Adventure,Biography",6.0,1048


## 3) Load TMDB 5000 and clean the data

In [27]:

df_tmdb = pd.read_csv(tmdb_csv)

# Convert date -> year
df_tmdb["release_date"] = pd.to_datetime(df_tmdb["release_date"], errors="coerce")
df_tmdb["release_year"] = df_tmdb["release_date"].dt.year

# Keep columns that are useful and not huge text blobs
cols_keep = [
    "id", "title", "budget", "revenue", "runtime", "release_year",
    "genres", "original_language", "popularity", "vote_average", "vote_count", "status"
]
df_tmdb = df_tmdb[cols_keep].copy()

# Numeric cleanup
for col in ["budget", "revenue", "runtime", "popularity", "vote_average", "vote_count"]:
    df_tmdb[col] = pd.to_numeric(df_tmdb[col], errors="coerce")

df_tmdb.head()


,id,title,budget,revenue,runtime,release_year,genres,original_language,popularity,vote_average,vote_count,status
0,19995,Avatar,237000000,2787965087,162.0,2009.0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",en,150.437577,7.2,11800,Released
1,285,Pirates of the Caribbean: At World's End,300000000,961000000,169.0,2007.0,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",en,139.082615,6.9,4500,Released
2,206647,Spectre,245000000,880674609,148.0,2015.0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",en,107.376788,6.3,4466,Released
3,49026,The Dark Knight Rises,250000000,1084939099,165.0,2012.0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",en,112.312950,7.6,9106,Released
4,49529,John Carter,260000000,284139100,132.0,2012.0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",en,43.926995,6.1,2124,Released


## 4) Feature engineering: primary genre

In [28]:

import ast

def extract_primary_genre(genres_str):
    if pd.isna(genres_str):
        return np.nan
    try:
        genres = ast.literal_eval(genres_str)
        if isinstance(genres, list) and len(genres) > 0:
            return genres[0].get("name")
    except Exception:
        return np.nan
    return np.nan

df_tmdb["primary_genre"] = df_tmdb["genres"].apply(extract_primary_genre)
df_tmdb["has_budget"] = df_tmdb["budget"].fillna(0).gt(0).astype(int)
df_tmdb["has_revenue"] = df_tmdb["revenue"].fillna(0).gt(0).astype(int)

df_tmdb[["title", "primary_genre", "budget", "revenue", "vote_average", "vote_count"]].head()


,title,primary_genre,budget,revenue,vote_average,vote_count
0,Avatar,Action,237000000,2787965087,7.2,11800
1,Pirates of the Caribbean: At World's End,Adventure,300000000,961000000,6.9,4500
2,Spectre,Action,245000000,880674609,6.3,4466
3,The Dark Knight Rises,Action,250000000,1084939099,7.6,9106
4,John Carter,Action,260000000,284139100,6.1,2124


## 5) Build the modeling dataset

In [29]:

# Basic quality filters:
# - Keep movies with known revenue and budget > 0
# - Keep reasonable runtimes
df_tmdb_clean = df_tmdb.copy()

df_tmdb_clean = df_tmdb_clean[df_tmdb_clean["budget"].gt(0) & df_tmdb_clean["revenue"].gt(0)]
df_tmdb_clean = df_tmdb_clean[df_tmdb_clean["runtime"].between(40, 240, inclusive="both") | df_tmdb_clean["runtime"].isna()]

# Add ROI. Keep it as a derived signal for EDA
# Do NOT use revenue-derived features when modeling revenue to avoid leakage.
df_tmdb_clean["roi"] = df_tmdb_clean["revenue"] / df_tmdb_clean["budget"]

df_tmdb_clean = df_tmdb_clean.dropna(subset=["release_year", "primary_genre", "original_language"])
df_tmdb_clean.reset_index(drop=True, inplace=True)

df_tmdb_clean.describe(include="all").T.head(12)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,3226.0,NaN,NaN,NaN,44791.378797,74641.301833,5.0,4952.75,11446.5,45271.25,417859.0
title,3226,3225,The Host,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
budget,3226.0,NaN,NaN,NaN,40676406.440174,44410056.186441,1.0,10500000.0,25000000.0,55000000.0,380000000.0
revenue,3226.0,NaN,NaN,NaN,121332909.17173,186363428.56631,5.0,17020041.75,55191503.0,146446330.5,2787965087.0
runtime,3226.0,NaN,NaN,NaN,110.608493,20.446736,41.0,96.0,107.0,121.0,238.0
release_year,3226.0,NaN,NaN,NaN,2001.689399,13.267396,1916.0,1998.0,2005.0,2010.0,2016.0
genres,3226,931,"[{""id"": 18, ""name"": ""Drama""}]",204,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_language,3226,27,en,3099,NaN,NaN,NaN,NaN,NaN,NaN,NaN
popularity,3226.0,NaN,NaN,NaN,29.05622,36.174533,0.019984,10.475636,20.421905,37.3508,875.581305
vote_average,3226.0,NaN,NaN,NaN,6.309516,0.873939,0.0,5.8,6.3,6.9,8.5


## 6) Save cleaned dataset movies_modeling_dataset.csv (saved to Colab files)

In [30]:

out_path = DATA_DIR / "movies_modeling_dataset.csv"
df_tmdb_clean.to_csv(out_path, index=False)
print(f"Saved cleaned dataset: {out_path} (rows={len(df_tmdb_clean):,}, cols={df_tmdb_clean.shape[1]})")


Saved cleaned dataset: /content/data/movies_modeling_dataset.csv (rows=3,226, cols=16)
